# 🧪 Lab 1: The Tiny Airport Dataset (Semantic Equivalence & Multi-Format Ingestion Foundation)

In this foundational, zero-dependency lab, we step onto the runway of semi-structured data architecture. We will generate a tiny, high-fidelity synthetic commercial travel dataset (< 1 MB) to establish our simulator environment. By representing the exact same business objects across four distinct shapes—JSON Lines, native XML, a cursed CSV with an embedded JSON string, and a clean typed DataFrame—we will prove the core engineering principle of semantic equivalence before moving toward Spark 4's VARIANT layer.

### Step 1: Initialize Local Workspace Coordinates
We spin up an isolated local Spark session and configure safe directory parameters for our multi-format airport terminal targets.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import os
import uuid

# Boot up a clean, standalone local Spark session
spark = SparkSession.builder.getOrCreate()

run_id = uuid.uuid4().hex[:8]
base_dir = f"/tmp/spark_spark4_flight_lab1_{run_id}"

print(f"Active Spark Core Engine: {spark.version}")
print(f"Workspace Runtime Target Directory: {base_dir}")

Active Spark Core Engine: 4.1.2
Workspace Runtime Target Directory: /tmp/spark_spark4_flight_lab1_8c67db99


### Step 2: Fabricate the Canonical Clean Analytical View
We establish our source of truth by building a typed Spark DataFrame representing the pristine commercial offers we wish every upstream provider sent us.

In [2]:
from decimal import Decimal

# Define explicit schema for our canonical reference dataset
canonical_schema = T.StructType([
    T.StructField("booking_id", T.StringType(), False),
    T.StructField("origin", T.StringType(), False),
    T.StructField("destination", T.StringType(), False),
    T.StructField("carrier", T.StringType(), False),
    T.StructField("fare_family", T.StringType(), False),
    T.StructField("total_price", T.DecimalType(10, 2), False),
    T.StructField("currency", T.StringType(), False),
    T.StructField("baggage_included", T.BooleanType(), False),
    T.StructField("passenger_count", T.IntegerType(), False)
])

# Generate two distinct, realistic flight offers using Decimal strings
flight_offers = [
    ("BKG-001", "MAD", "BCN", "IB", "EconomyLight", Decimal("87.50"), "EUR", True, 1),
    ("BKG-002", "CDG", "JFK", "AF", "EconomyClassic", Decimal("520.00"), "USD", True, 2)
]

canonical_df = spark.createDataFrame(flight_offers, canonical_schema)
print("=== CANONICAL TYPED DATAFRAME LAYER ===")
canonical_df.show(truncate=False)
canonical_df.printSchema()

=== CANONICAL TYPED DATAFRAME LAYER ===
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+
|booking_id|origin|destination|carrier|fare_family   |total_price|currency|baggage_included|passenger_count|
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+
|BKG-001   |MAD   |BCN        |IB     |EconomyLight  |87.50      |EUR     |true            |1              |
|BKG-002   |CDG   |JFK        |AF     |EconomyClassic|520.00     |USD     |true            |2              |
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+

root
 |-- booking_id: string (nullable = false)
 |-- origin: string (nullable = false)
 |-- destination: string (nullable = false)
 |-- carrier: string (nullable = false)
 |-- fare_family: string (nullable = false)
 |-- total_price: decimal(10,2) (nullable = false)
 |-- currency: string (nullable = false

### Step 3: Emit Data into Alternative Transport Shapes
To establish our multi-representation testing arena, we write the data out into three separate filesystem nodes: JSON Lines, Spark 4 Native XML, and a hybrid CSV containing an encapsulated JSON string payload.

In [3]:
os.makedirs(base_dir, exist_ok=True)
json_path = f"{base_dir}/json_lines"
xml_path = f"{base_dir}/xml_tags"
csv_path = f"{base_dir}/csv_embedded"

# Shape 1: Standard Event-style JSON Lines
canonical_df.write.mode("overwrite").json(json_path)

# Shape 2: Native XML under Spark 4 rules using explicit row identifiers
canonical_df.write.mode("overwrite").format("xml").option("rowTag", "booking").save(xml_path)

# Shape 3: Cursed but mandatory CSV containing an embedded JSON string wrapper
csv_embedded_df = canonical_df.select(
    "booking_id",
    F.to_json(F.struct("origin", "destination", "carrier", "fare_family", "total_price", "currency", "baggage_included", "passenger_count")).alias("embedded_payload")
)
csv_embedded_df.write.mode("overwrite").option("header", "true").csv(csv_path)

print("All transport shapes successfully saved to filesystem layout.")

All transport shapes successfully saved to filesystem layout.


### Step 4: Interrogate Shape 1 — The JSON Lines Ingestion Node
We read back our JSON Lines layer and extract our essential operational travel metrics.

In [4]:
read_json_df = spark.read.json(json_path)

processed_json_df = (
    read_json_df
    .withColumn("route_type", F.when((F.col("origin") == "MAD") & (F.col("destination") == "BCN"), F.lit("short-haul")).otherwise(F.lit("long-haul")))
    .select("booking_id", "origin", "destination", "carrier", "fare_family", F.col("total_price").cast("decimal(10,2)").alias("total_price"), "currency", "baggage_included", "passenger_count", "route_type")
)

print("=== INGESTED RECOVERY FROM JSON LINES ===")
processed_json_df.show(truncate=False)

=== INGESTED RECOVERY FROM JSON LINES ===
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+----------+
|booking_id|origin|destination|carrier|fare_family   |total_price|currency|baggage_included|passenger_count|route_type|
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+----------+
|BKG-002   |CDG   |JFK        |AF     |EconomyClassic|520.00     |USD     |true            |2              |long-haul |
|BKG-001   |MAD   |BCN        |IB     |EconomyLight  |87.50      |EUR     |true            |1              |short-haul|
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+----------+



### Step 5: Interrogate Shape 2 — The Built-In Spark 4 XML Ingest
We leverage Spark 4's native XML datasource capability, passing our targeted row tag directly to reconstruct our nested offer parameters cleanly.

In [5]:
read_xml_df = spark.read.format("xml").option("rowTag", "booking").load(xml_path)

processed_xml_df = (
    read_xml_df
    .withColumn("route_type", F.when((F.col("origin") == "MAD") & (F.col("destination") == "BCN"), F.lit("short-haul")).otherwise(F.lit("long-haul")))
    .select("booking_id", "origin", "destination", "carrier", "fare_family", F.col("total_price").cast("decimal(10,2)").alias("total_price"), "currency", "baggage_included", "passenger_count", "route_type")
)

print("=== INGESTED RECOVERY FROM NATIVE XML SOURCE ===")
processed_xml_df.show(truncate=False)

=== INGESTED RECOVERY FROM NATIVE XML SOURCE ===
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+----------+
|booking_id|origin|destination|carrier|fare_family   |total_price|currency|baggage_included|passenger_count|route_type|
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+----------+
|BKG-002   |CDG   |JFK        |AF     |EconomyClassic|520.00     |USD     |true            |2              |long-haul |
|BKG-001   |MAD   |BCN        |IB     |EconomyLight  |87.50      |EUR     |true            |1              |short-haul|
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+----------+



### Step 6: Interrogate Shape 3 — Unpacking the Cursed Embedded String Layer
We pull the raw hybrid CSV rows and dynamically evaluate the internal text payload using inline json functions to uncover the hidden business metrics.

In [6]:
read_csv_df = spark.read.option("header", "true").csv(csv_path)

# Reconstruct structural representation from string text payload
json_payload_schema = T.StructType([
    T.StructField("origin", T.StringType(), True),
    T.StructField("destination", T.StringType(), True),
    T.StructField("carrier", T.StringType(), True),
    T.StructField("fare_family", T.StringType(), True),
    T.StructField("total_price", T.DecimalType(10, 2), True),
    T.StructField("currency", T.StringType(), True),
    T.StructField("baggage_included", T.BooleanType(), True),
    T.StructField("passenger_count", T.IntegerType(), True)
])

processed_csv_df = (
    read_csv_df
    .withColumn("parsed_struct", F.from_json(F.col("embedded_payload"), json_payload_schema))
    .withColumn("route_type", F.when((F.col("parsed_struct.origin") == "MAD") & (F.col("parsed_struct.destination") == "BCN"), F.lit("short-haul")).otherwise(F.lit("long-haul")))
    .select( 
        "booking_id",
        F.col("parsed_struct.origin").alias("origin"),
        F.col("parsed_struct.destination").alias("destination"),
        F.col("parsed_struct.carrier").alias("carrier"),
        F.col("parsed_struct.fare_family").alias("fare_family"),
        F.col("parsed_struct.total_price").alias("total_price"),
        F.col("parsed_struct.currency").alias("currency"),
        F.col("parsed_struct.baggage_included").alias("baggage_included"),
        F.col("parsed_struct.passenger_count").alias("passenger_count"),
        "route_type"
    )
)

print("=== INGESTED RECOVERY FROM EMBEDDED CSV PAYLOAD ===")
processed_csv_df.show(truncate=False)

=== INGESTED RECOVERY FROM EMBEDDED CSV PAYLOAD ===
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+----------+
|booking_id|origin|destination|carrier|fare_family   |total_price|currency|baggage_included|passenger_count|route_type|
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+----------+
|BKG-002   |CDG   |JFK        |AF     |EconomyClassic|520.00     |USD     |true            |2              |long-haul |
|BKG-001   |MAD   |BCN        |IB     |EconomyLight  |87.50      |EUR     |true            |1              |short-haul|
+----------+------+-----------+-------+--------------+-----------+--------+----------------+---------------+----------+



### Step 7: Mathematically Validate Semantic Equivalence
We compare the final computed fields across all four transport representations to confirm they express identical business answers.

In [7]:
json_check = processed_json_df.filter(F.col("booking_id") == "BKG-001").select("total_price", "currency").collect()[0]
xml_check = processed_xml_df.filter(F.col("booking_id") == "BKG-001").select("total_price", "currency").collect()[0]
csv_check = processed_csv_df.filter(F.col("booking_id") == "BKG-001").select("total_price", "currency").collect()[0]
canonical_check = canonical_df.filter(F.col("booking_id") == "BKG-001").select("total_price", "currency").collect()[0]

print("=== SEMANTIC EQUIVALENCE REPORT FOR MAD -> BCN ===")
print(f"Canonical DataFrame Value : {canonical_check['total_price']} {canonical_check['currency']}")
print(f"JSON Lines Node Value     : {json_check['total_price']} {json_check['currency']}")
print(f"Native XML Layer Value    : {xml_check['total_price']} {xml_check['currency']}")
print(f"CSV Embedded Payload Value: {csv_check['total_price']} {csv_check['currency']}")

assert canonical_check == json_check == xml_check == csv_check, "Validation Failure: Structural shapes disagree!"
print("\n[PASSED] Core validation target complete. Semantic identity successfully verified across shapes.")

=== SEMANTIC EQUIVALENCE REPORT FOR MAD -> BCN ===
Canonical DataFrame Value : 87.50 EUR
JSON Lines Node Value     : 87.50 EUR
Native XML Layer Value    : 87.50 EUR
CSV Embedded Payload Value: 87.50 EUR

[PASSED] Core validation target complete. Semantic identity successfully verified across shapes.


## 📊 Post-Lab Analysis: Evidence from the Engine Room

Evaluating the empirical outputs from this foundational lab provides concrete proof of the strict boundary line separating transport data containers from unified business meaning. By analyzing row distribution patterns, lazy execution timelines, and strict type constraints, we can cleanly deconstruct the architectural illusion of "automatic" data ingestion.

### 1. Transport Containers Are Not Business Models
This lab successfully demonstrates the core engineering principle of **semantic equivalence**: a data product is defined by its business truth, not its transient structural packaging.
* **Identical Business Truth**: The single flight transaction `BKG-001` preserved its precise price metric of **87.50 EUR** across all four parallel representations.
* **Decoupled Envelopes**: Whether wrapped in event-style JSON Lines, packaged inside an XML tag tree, buried as an escaped text string inside a CSV column, or presented in a typed DataFrame, the commercial payload remained identical .
* **The Takeaway**: Transport mediums are highly volatile operational variables; true architecture is defined by what your pipeline preserves, parses, and promotes.

### 2. The Shuffled Passenger Manifest (Distributed Row Reordering)
An unexpected but highly educational shift occurred when reading the streams back from disk: `BKG-002` (the long-haul offer) materialized *before* `BKG-001`, completely inverting our initial insertion order.
* **Parallel File Scans**: Spark naturally partitions data and writes dataframes into multi-part transactional files across distributed nodes.
* **Non-Deterministic Listing**: When reading a directory back into memory, Spark performs an un-sequenced parallel file-listing operation.
* **The Takeaway**: Rows inside an analytical data lake are completely fluid; strict sequencing is never guaranteed unless a global sort constraint (`orderBy`) is explicitly declared.

### 3. The Lazy Evaluation Tax Exposed
A profound timing delta was captured inside the engine logs: defining, parsing, and building the dataframes took less than 2 seconds combined, while the final validation pass in Step 7 consumed the vast majority of the execution runtime.
* **Blueprint Construction**: Spark operates entirely on **lazy evaluation**. Steps 4, 5, and 6 did not actually process any rows; they merely registered metadata transformation blueprints inside the logical plan parser.
* **Action-Driven Heartbeat**: The moment our assertion script called `.collect()` in Step 7, a live Spark action was triggered. The engine was forced to compile the optimized physical plans, serialize tasks, distribute them across workers, and execute all four ingestion tracks concurrently.
* **The Takeaway**: In streaming and batch architectures, the validation phase pays the accumulated computational tax for the entire processing runway.

### 4. Floating-Point Improvisation vs. Strict Contracts
The initial type collision during development provided a critical lesson regarding financial data governance.
* **The Imprecision Trap**: Primitive binary floating-point types (`floats` and `doubles`) are inherently imprecise, making them completely unacceptable for monetary and tax reconciliation.
* **Strict Schema Enforcement**: By strictly mapping fields to `DecimalType(10, 2)` and supplying explicit `decimal.Decimal` objects, we enforced absolute fixed-point precision at the gateway.
* **The Takeaway**: Schema contracts must enforce strict mathematical types early to prevent data type widening from corrupting downstream gold serving layers.
